Damped Newton method for chi=30
Start Newton method after 9 RG steps.
Damping with newton_step=0.5 is activated a couple of times (e.g. for i=2)
Here I ran up to i=15, reaching fp_error =5.4646911693865845e-9

In [1]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 2e-5 #6e-6
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.03361407516010624 and became 0.0. Index CartesianIndex(1, 16, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -0.012345916839578114 and became -1.8524388175538877e-9. Index CartesianIndex(15, 2, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.004606248308129038 and became 0.0. Index CartesianIndex(17, 3, 15, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.0022689612252432567 and became 0.0. Index CartesianIndex(15, 2, 19, 2) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It wa

In [14]:
for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

1 [1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
5 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
6 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
7 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
8 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
9 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
10 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]
11 [14 16; 15 15; 14 16; 15 15][0 1; 0 1; 0 1; 0 1]


In [3]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

In [4]:
A[1] = truncate_blocks(traj[10], trunc_shape)
for i in 1:30
    println("i=",i)
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    e0, RAshape = fp_error_with_shape(A[i],accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RAshape)
    flush(stdout)
    if A[i].shape != RAshape
        throw(ErrorException("shapes unequal"))
    end
    deltaA[i] = newton_correction(A[i], 10, accepted_elements[i], gilt_pars; trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]
        Anew, accepted_elements_new = fix_discrete_gauge(Anew; tol = 1e-7);
        enew, RAnewshape = fp_error_with_shape(Anew, accepted_elements_new, gilt_pars; trunc_shape = trunc_shape)
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnewshape)
        if enew < e0 && Anew.shape == RAnewshape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
||R(A[i])-A[i]||= 0.02666307971722672
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  12 eigenvalues converged
│ *  norm of residuals = (4.536904297041038e-51, 4.1038907418447935e-37, 2.7886272283063356e-36, 4.3571063208544875e-30, 2.598515850569977e-26, 1.822636006219755e-25, 6.357686076451218e-24, 6.357686076451218e-24, 6.037083060003069e-14, 6.037083060003069e-14, 1.276225496708274e-15, 1.276225496708274e-15)
└ *  number of operations = 58


EIGENVALUES (INITIAL):
1.9940156749476425 + 0.0im
-0.9876526041869125 + 0.0im
-0.9781342392360158 + 0.0im
0.6648548847796065 + 0.0im
0.5904552872262573 + 0.0im
-0.5202682224731843 + 0.0im
-0.01511631635212536 + 0.48500839800741324im
-0.01511631635212536 - 0.48500839800741324im
-0.011765021779511314 + 0.30492000531645425im
-0.011765021779511314 - 0.30492000531645425im
-0.1634984039821701 + 0.2563774446848964im
-0.1634984039821701 - 0.2563774446848964im
||deltaA[i]||= 0.03138538746683645
newton_step= 1.0
fp_error= 0.015485534245547983
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=2
||R(A[i])-A[i]||= 0.015485534245547983
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  10 eigenvalues converged
│ *  norm of residuals = (0.0, 1.6516604544237656e-37, 3.726670599146055e-37, 1.0845406088789873e-36, 2.03652648409521e-28, 4.003593548639711e-26, 4.003593548639711e-26, 4.460930557542185e-20, 7.287960644493556e-15, 7.287960644493556e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9966275310797357 + 0.0im
-0.9899522174979257 + 0.0im
-0.980536321772077 + 0.0im
0.8277433990564459 + 0.0im
0.6016660935246934 + 0.0im
-0.007805183909141257 + 0.5217133413371406im
-0.007805183909141257 - 0.5217133413371406im
-0.4025251203084254 + 0.0im
0.14209643079121548 + 0.2592997037104018im
0.14209643079121548 - 0.2592997037104018im
||deltaA[i]||= 0.05251075003128929
newton_step= 1.0
fp_error= 0.1106877704425304
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
newton_step= 0.5
fp_error= 0.008313332439580412
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=3
||R(A[i])-A[i]||= 0.008313332439580412
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


...

┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (1.2685172069549649e-53, 2.4441038033459016e-38, 1.750579272556382e-38, 8.80053827986765e-31, 7.682971109214291e-29, 4.9799088227147454e-26, 4.9799088227147454e-26, 1.6047614976527022e-18, 1.0199732311563177e-16, 1.2642237451982744e-15, 1.2642237451982744e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9983678020064466 + 0.0im
-0.994621900518146 + 0.0im
-0.986657730575735 + 0.0im
0.6423199687079822 + 0.0im
0.5967840379808141 + 0.0im
-0.01002253688623498 + 0.5021858108389686im
-0.01002253688623498 - 0.5021858108389686im
0.37491660137106186 + 0.0im
-0.3477999311630843 + 0.0im
-0.1485695879659018 + 0.24544795624336052im
-0.1485695879659018 - 0.24544795624336052im
||deltaA[i]||= 1.94335966594694e-7
newton_step= 1.0
fp_error= 2.5345481427753866e-8
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=14
||R(A[i])-A[i]||= 2.5345481427753866e-8
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  11 eigenvalues converged
│ *  norm of residuals = (2.2241668529736493e-53, 5.6962562981980105e-39, 7.613081111279799e-39, 9.836479881466265e-31, 5.825767387756571e-29, 2.767171530866382e-26, 2.767171530866382e-26, 5.2088290243767365e-19, 1.753332100557321e-17, 3.505385992084837e-15, 3.505385992084837e-15)
└ *  number of operations = 59


EIGENVALUES (INITIAL):
1.9983677361146233 + 0.0im
-0.9946214881278622 + 0.0im
-0.9866577882756384 + 0.0im
0.6425288023914612 + 0.0im
0.596716770707597 + 0.0im
-0.010016364422044708 + 0.5022207178433005im
-0.010016364422044708 - 0.5022207178433005im
0.37473929033920006 + 0.0im
-0.34785626188343427 + 0.0im
-0.1485536348206398 + 0.24548417984545254im
-0.1485536348206398 - 0.24548417984545254im
||deltaA[i]||= 6.027560244963155e-8
newton_step= 1.0
fp_error= 5.4646911693865845e-9
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
i=15
||R(A[i])-A[i]||= 5.4646911693865845e-9
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]


LoadError: InterruptException: